In [ ]:
import pandas as pd
import networkx as nx
from datetime import datetime, timedelta
from tqdm import tqdm

from pathlib import Path

# 노트북을 어느 하위 폴더에서 실행해도 저장소 루트를 찾습니다.
_current_dir = Path.cwd().resolve()
PROJECT_ROOT = next(
    path for path in (_current_dir, *_current_dir.parents)
    if (path / "notebooks").is_dir() and (path / "data").is_dir()
)
DATA_DIR = PROJECT_ROOT / "data"
RESULTS_DIR = PROJECT_ROOT / "results"


In [ ]:
input_path = DATA_DIR / "processed" / "solana" / "SOL_2020-10-01_2020-12-31_TRX.csv"
df = pd.read_csv(input_path)
df["timestamp"] = pd.to_datetime(df["timestamp"])


In [14]:
# timestamp를 datetime으로 변환
df['timestamp'] = pd.to_datetime(df['timestamp'])

# 시작일과 종료일 설정
start_date = df['timestamp'].min().date()
end_date = df['timestamp'].max().date()

# 결과를 저장할 리스트
results = []

# 진행 상황을 표시할 tqdm 객체 생성
total_days = (end_date - start_date).days + 1
pbar = tqdm(total=total_days, desc="Processing days")

# 각 날짜별로 계산
current_date = start_date
while current_date <= end_date:
    # 해당 날짜의 데이터만 필터링
    daily_df = df[df['timestamp'].dt.date == current_date]
    
    # 그래프 생성
    G = nx.from_pandas_edgelist(daily_df, 'sender_address', 'receiver_address', edge_attr='amount', create_using=nx.DiGraph())
    G.remove_edges_from(nx.selfloop_edges(G))
    
    # 그래프가 비어있지 않은 경우에만 계산
    if G.number_of_nodes() > 0:
        # Core decomposition
        core_numbers = nx.core_number(G)
        max_core = max(core_numbers.values()) if core_numbers else 0
        
        try:
            adhesion = nx.edge_connectivity(G)
        except nx.NetworkXError:
            adhesion = 0  # For disconnected graphs
        
        # Cohesion (node connectivity)
        try:
            cohesion = nx.node_connectivity(G)
        except nx.NetworkXError:
            cohesion = 0  # For disconnected graphs
        
        # 결과 저장
        results.append({
            'date': current_date,
            'max_core_number': max_core,
            'adhesion': adhesion,
            'cohesion': cohesion
        })
    
    # 다음 날짜로 이동
    current_date += timedelta(days=1)

    # 진행 상황 업데이트
    pbar.update(1)

# 진행 바 닫기
pbar.close()

# 결과를 데이터프레임으로 변환
result_df = pd.DataFrame(results)

# 결과 출력
print(result_df)

Processing days: 100%|███████████████████████████████████████████████████████████████| 92/92 [02:12<00:00,  1.44s/it]

          date  max_core_number  adhesion  cohesion
0   2020-10-01                4         0         0
1   2020-10-02                4         0         0
2   2020-10-03                4         0         0
3   2020-10-04                4         0         0
4   2020-10-05                4         0         0
..         ...              ...       ...       ...
87  2020-12-27                3         0         0
88  2020-12-28                3         0         0
89  2020-12-29                3         0         0
90  2020-12-30                3         0         0
91  2020-12-31                3         0         0

[92 rows x 4 columns]


In [15]:
result_df

,date,max_core_number,adhesion,cohesion
0,2020-10-01,4,0,0
1,2020-10-02,4,0,0
2,2020-10-03,4,0,0
3,2020-10-04,4,0,0
4,2020-10-05,4,0,0
...,...,...,...,...
87,2020-12-27,3,0,0
88,2020-12-28,3,0,0
89,2020-12-29,3,0,0
90,2020-12-30,3,0,0


In [ ]:
output_path = RESULTS_DIR / "solana" / "network_metrics" / "SOL_2020-10-01_to_2020-12-31_connectivity.csv"
output_path.parent.mkdir(parents=True, exist_ok=True)
result_df.to_csv(output_path, index=False)


In [7]:
# 2021년 2차 계산항목 계산중